# Projeto Final - Validação de MVPs (1, 2 e 3)
Este notebook orquestra o pipeline de dados:
1. **MVP 1:** Leitura automatizada e tolerante a falhas (CSV/XLSX).
2. **MVP 2:** Profiling estrutural (sem alteração de dados originais).
3. **MVP 3:** Comparação entre dados necessários (finalidade) e dados encontrados (base).

In [1]:
# O autoreload garante que se você alterar os arquivos .py, o notebook atualiza automaticamente
%load_ext autoreload
%autoreload 2

import json
from src.loader import load_data
from src.profiler import profile_data
from src.comparator import check_required_data, print_analise_status

print("Módulos carregados com sucesso!")

Módulos carregados com sucesso!


## Execução do Pipeline com Bases Reais
Neste teste, validamos o sistema contra arquivos reais baixados de fontes distintas para garantir a resiliência do sistema de aliases e do carregador.

In [ ]:
# Lista dos arquivos reais da pasta data/exemplo/
arquivos_teste = [
    "data/exemplo/vendas_kaggle.csv",
    "data/exemplo/vendas_governo.csv",
    "data/exemplo/erp.csv"
]

for caminho in arquivos_teste:
    print(f"🚀 INICIANDO TESTE COM: {caminho}")
    try:
        # 1. Carrega os dados (MVP 1)
        carga = load_data(caminho)
        df = carga["df"]
        print(f"   [Leitura] Extensão: {carga['metadata']['extensao']} | Encoding: {carga['metadata']['encoding']} | Delimitador: '{carga['metadata']['delimitador']}'")
        
        # 2. Gera o profiling (MVP 2)
        perfil = profile_data(df)
        
        # 3. Compara com os requisitos (MVP 3)
        resultado = check_required_data(perfil, "faturamento_por_regiao")
        
        # 4. Exibe o resultado formatado
        print_analise_status(resultado)
        
    except FileNotFoundError:
        print(f"⚠️ ARQUIVO NÃO ENCONTRADO: {caminho}")
        print("-> Baixe uma base de teste, renomeie com este nome e salve na pasta 'data/exemplo/'.")
    except Exception as e:
        print(f"❌ Erro ao processar a base: {e}")
        
    print("\n" + "="*60 + "\n")

🚀 INICIANDO TESTE COM: data/exemplo/vendas_kaggle.csv
FINALIDADE
Faturamento por Região

DADOS NECESSÁRIOS

data_venda
→ Encontrado
→ Coluna: Sale_Date

regiao
→ Encontrado
→ Coluna: Region

valor_venda
→ Encontrado
→ Coluna: Sales_Amount

DADOS ADICIONAIS

Product_ID
Sales_Rep
Quantity_Sold
Product_Category
Unit_Cost
Unit_Price
Customer_Type
Discount
Payment_Method
Sales_Channel
Region_and_Sales_Rep

STATUS

APTA


🚀 INICIANDO TESTE COM: data/exemplo/vendas_governo.csv
FINALIDADE
Faturamento por Região

DADOS NECESSÁRIOS

data_venda
→ Encontrado
→ Coluna: ANO

regiao
→ Encontrado
→ Coluna: GRANDE REGIÃO

valor_venda
→ Encontrado
→ Coluna: VENDAS

DADOS ADICIONAIS

MÊS
UNIDADE DA FEDERAÇÃO
PRODUTO

STATUS

APTA


🚀 INICIANDO TESTE COM: data/exemplo/erp.csv
FINALIDADE
Faturamento por Região

DADOS NECESSÁRIOS

data_venda
→ Encontrado
→ Coluna: order_date

regiao
→ NÃO encontrado

valor_venda
→ NÃO encontrado

DADOS ADICIONAIS

sales_id
product_id
customer_id
channel_id
qty
unit_price
di

## Inspeção Profunda (Opcional)
Visualização do dicionário JSON bruto gerado pelo MVP 2 para auditoria.

In [3]:
caminho_inspecao = "data/exemplo/vendas_kaggle.csv"

try:
    df = load_data(caminho_inspecao)
    perfil = profile_data(df)
    
    print(f"Radiografia da base: {caminho_inspecao}\n")
    # Mostra apenas os primeiros 1000 caracteres para não travar o notebook com bases gigantes
    json_saida = json.dumps(perfil, indent=4, ensure_ascii=False)
    if len(json_saida) > 1000:
        print(json_saida[:1000] + "\n\n... [JSON TRUNCADO PARA FACILITAR LEITURA]")
    else:
        print(json_saida)
except FileNotFoundError:
    print("Para inspecionar o JSON, garanta que o arquivo acima existe.")

Radiografia da base: data/exemplo/vendas_kaggle.csv

{
    "geral": {
        "linhas": 1000,
        "colunas": 14,
        "duplicadas": 0
    },
    "colunas": [
        {
            "nome": "Product_ID",
            "tipo": "int64",
            "total": 1000,
            "preenchidos": 1000,
            "nulos": 0,
            "percentual_nulos": 0.0,
            "valores_distintos": 100
        },
        {
            "nome": "Sale_Date",
            "tipo": "object",
            "total": 1000,
            "preenchidos": 1000,
            "nulos": 0,
            "percentual_nulos": 0.0,
            "valores_distintos": 340
        },
        {
            "nome": "Sales_Rep",
            "tipo": "object",
            "total": 1000,
            "preenchidos": 1000,
            "nulos": 0,
            "percentual_nulos": 0.0,
            "valores_distintos": 5
        },
        {
            "nome": "Region",
            "tipo": "object",
            "total": 1000,
            "p